In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

# Load the enhanced_student_habits_performance_dataset
df = pd.read_csv(r'c:\vscode\CVprojects\kenexaiHackathon\datasets\studentPerformance.csv')

print("=" * 70)
print("STUDENT HABITS & PERFORMANCE DATASET - PREPROCESSING PIPELINE")
print("=" * 70)

# Step 1: Check dataset structure
print("\n1. DATASET STRUCTURE")
print("-" * 70)
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\n\nDataset Info:")
print(df.info())
print(f"\n\nBasic Statistics:")
print(df.describe())

STUDENT HABITS & PERFORMANCE DATASET - PREPROCESSING PIPELINE

1. DATASET STRUCTURE
----------------------------------------------------------------------
Shape: (80000, 31)

First few rows:
   student_id  age  gender             major  study_hours_per_day  \
0      100000   26    Male  Computer Science             7.645367   
1      100001   28    Male              Arts             5.700000   
2      100002   17    Male              Arts             2.400000   
3      100003   27   Other        Psychology             3.400000   
4      100004   25  Female          Business             4.700000   

   social_media_hours  netflix_hours part_time_job  attendance_percentage  \
0                 3.0            0.1           Yes                   70.3   
1                 0.5            0.4            No                   88.4   
2                 4.2            0.7            No                   82.1   
3                 4.6            2.3           Yes                   79.3   
4        

In [13]:
# Step 2: Clean column names
print("\n2. CLEAN COLUMN NAMES")
print("-" * 70)

df.columns = (df.columns.str.strip()
              .str.lower()
              .str.replace(r'[^a-z0-9]+', '_', regex=True)
              .str.strip('_'))

print("Cleaned column names:")
print(df.columns.tolist())
print(f"\nDataset shape: {df.shape}")


2. CLEAN COLUMN NAMES
----------------------------------------------------------------------
Cleaned column names:
['student_id', 'age', 'gender', 'major', 'study_hours_per_day', 'social_media_hours', 'netflix_hours', 'part_time_job', 'attendance_percentage', 'sleep_hours', 'diet_quality', 'exercise_frequency', 'parental_education_level', 'internet_quality', 'mental_health_rating', 'extracurricular_participation', 'previous_gpa', 'semester', 'stress_level', 'dropout_risk', 'social_activity', 'screen_time', 'study_environment', 'access_to_tutoring', 'family_income_range', 'parental_support_level', 'motivation_level', 'exam_anxiety_score', 'learning_style', 'time_management_score', 'exam_score']

Dataset shape: (80000, 31)


In [14]:
# Step 3: Check missing values and duplicates
print("\n3. CHECK MISSING VALUES & DUPLICATES")
print("-" * 70)

print("Missing values per column:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0].sort_values(ascending=False))
else:
    print("No missing values found")

print(f"\n\nMissing value percentage:")
missing_pct = (df.isnull().sum() / len(df)) * 100
if missing_pct.sum() > 0:
    print(missing_pct[missing_pct > 0].sort_values(ascending=False))
else:
    print("No missing values")

print(f"\n\nDuplicate records:")
duplicates = df.duplicated().sum()
print(f"Total duplicates: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicates found")


3. CHECK MISSING VALUES & DUPLICATES
----------------------------------------------------------------------
Missing values per column:
No missing values found


Missing value percentage:
No missing values


Duplicate records:
Total duplicates: 0
No duplicates found


In [15]:
# Step 4: Handle missing values
print("\n4. HANDLE MISSING VALUES")
print("-" * 70)

# Get numeric and categorical columns
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Fill numeric columns with median
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled {col} with median: {median_val}")

# Fill categorical columns with mode
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0] if not df[col].mode().empty else 'Unknown'
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled {col} with mode: {mode_val}")

print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")
print(f"Dataset shape: {df.shape}")


4. HANDLE MISSING VALUES
----------------------------------------------------------------------

Total missing values remaining: 0
Dataset shape: (80000, 31)


In [16]:
# Step 5: Fix categorical columns
print("\n5. FIX CATEGORICAL COLUMNS")
print("-" * 70)

# Standardize and convert to category type
categorical_to_fix = ['gender', 'major', 'diet_quality', 'parental_education_level',
                      'internet_quality', 'learning_style', 'study_environment', 
                      'family_income_range']

for col in categorical_to_fix:
    if col in df.columns:
        # Clean the values: strip whitespace and convert to lowercase
        df[col] = df[col].astype(str).str.strip().str.lower()
        # Convert to category type
        df[col] = df[col].astype('category')
        print(f"✓ {col}: {df[col].nunique()} categories")

print(f"\nCategorical columns converted to category dtype")


5. FIX CATEGORICAL COLUMNS
----------------------------------------------------------------------
✓ gender: 3 categories
✓ major: 6 categories
✓ diet_quality: 3 categories
✓ parental_education_level: 5 categories
✓ internet_quality: 3 categories
✓ learning_style: 4 categories
✓ study_environment: 5 categories
✓ family_income_range: 3 categories

Categorical columns converted to category dtype


In [17]:
# Step 6: Standardize binary columns (Yes/No to 1/0)
print("\n6. STANDARDIZE BINARY COLUMNS")
print("-" * 70)

binary_cols = ['part_time_job', 'extracurricular_participation', 'access_to_tutoring', 
               'dropout_risk']

for col in binary_cols:
    if col in df.columns:
        # Check current values
        unique_vals = df[col].unique()
        print(f"{col} unique values before: {unique_vals}")
        
        # Convert Yes/No or True/False to 1/0
        df[col] = df[col].astype(str).str.lower().map({'yes': 1, 'true': 1, '1': 1, 
                                                        'no': 0, 'false': 0, '0': 0})
        
        # For any remaining non-matching values, convert to 0
        df[col] = df[col].fillna(0).astype(int)
        
        print(f"  After: {sorted(df[col].unique())}")

print(f"\nBinary columns standardized to 1/0")


6. STANDARDIZE BINARY COLUMNS
----------------------------------------------------------------------
part_time_job unique values before: ['Yes' 'No']
  After: [0, 1]
extracurricular_participation unique values before: ['Yes' 'No']
  After: [0, 1]
access_to_tutoring unique values before: ['Yes' 'No']
  After: [0, 1]
dropout_risk unique values before: ['No' 'Yes']
  After: [0, 1]

Binary columns standardized to 1/0


In [18]:
# Step 7: Outlier detection using IQR method
print("\n7. OUTLIER DETECTION (IQR METHOD)")
print("-" * 70)

outlier_cols = ['study_hours_per_day', 'sleep_hours', 'screen_time', 
                'social_media_hours', 'exam_score', 'previous_gpa']

outliers_count = 0
for col in outlier_cols:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers_in_col = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        if outliers_in_col > 0:
            print(f"{col}: {outliers_in_col} outliers detected")
            outliers_count += outliers_in_col
            
            # Cap outliers instead of removing
            df[col] = df[col].clip(lower_bound, upper_bound)

print(f"\nTotal outliers detected: {outliers_count}")
print("Outliers capped to min/max bounds (not removed)")
print(f"Dataset shape: {df.shape}")


7. OUTLIER DETECTION (IQR METHOD)
----------------------------------------------------------------------
study_hours_per_day: 346 outliers detected
sleep_hours: 284 outliers detected
screen_time: 356 outliers detected
exam_score: 467 outliers detected
previous_gpa: 201 outliers detected

Total outliers detected: 1654
Outliers capped to min/max bounds (not removed)
Dataset shape: (80000, 31)


In [19]:
# Step 8: Feature Engineering
print("\n8. FEATURE ENGINEERING")
print("-" * 70)

# Feature 1: Total screen time
if 'social_media_hours' in df.columns and 'netflix_hours' in df.columns:
    df['total_screen_time'] = df['social_media_hours'] + df['netflix_hours']
    print("✓ Created: total_screen_time = social_media_hours + netflix_hours")

# Feature 2: Productivity score
if 'study_hours_per_day' in df.columns and 'screen_time' in df.columns:
    # Avoid division by zero
    df['productivity_score'] = df['study_hours_per_day'] / (df['screen_time'] + 0.1)
    print("✓ Created: productivity_score = study_hours_per_day / screen_time")

# Feature 3: Wellbeing score
if all(col in df.columns for col in ['sleep_hours', 'exercise_frequency', 'mental_health_rating']):
    df['wellbeing_score'] = (df['sleep_hours'] + df['exercise_frequency'] + df['mental_health_rating']) / 3
    print("✓ Created: wellbeing_score = (sleep_hours + exercise_frequency + mental_health_rating)/3")

print(f"\nNew features created. Dataset shape: {df.shape}")
print(f"New features: total_screen_time, productivity_score, wellbeing_score")


8. FEATURE ENGINEERING
----------------------------------------------------------------------
✓ Created: total_screen_time = social_media_hours + netflix_hours
✓ Created: productivity_score = study_hours_per_day / screen_time
✓ Created: wellbeing_score = (sleep_hours + exercise_frequency + mental_health_rating)/3

New features created. Dataset shape: (80000, 34)
New features: total_screen_time, productivity_score, wellbeing_score


In [20]:
# Step 9: Normalize important numeric features
print("\n9. NORMALIZE NUMERIC FEATURES")
print("-" * 70)

normalize_cols = ['study_hours_per_day', 'attendance_percentage', 'sleep_hours', 
                  'stress_level', 'exam_anxiety_score', 'exam_score']

# Filter to only columns that exist in dataset
normalize_cols = [col for col in normalize_cols if col in df.columns]

if normalize_cols:
    scaler = MinMaxScaler()
    df[normalize_cols] = scaler.fit_transform(df[normalize_cols])
    print(f"Normalized {len(normalize_cols)} columns to [0, 1] range")
    print(f"Columns normalized: {normalize_cols}")
    
    print(f"\nNormalized data statistics:")
    print(df[normalize_cols].describe())
else:
    print("No columns to normalize found in dataset")


9. NORMALIZE NUMERIC FEATURES
----------------------------------------------------------------------
Normalized 6 columns to [0, 1] range
Columns normalized: ['study_hours_per_day', 'attendance_percentage', 'sleep_hours', 'stress_level', 'exam_anxiety_score', 'exam_score']

Normalized data statistics:
       study_hours_per_day  attendance_percentage   sleep_hours  stress_level  \
count         80000.000000           80000.000000  80000.000000  80000.000000   
mean              0.436821               0.499465      0.430833      0.445831   
std               0.209004               0.288884      0.208959      0.217019   
min               0.000000               0.000000      0.000000      0.000000   
25%               0.293194               0.250000      0.285714      0.288889   
50%               0.432002               0.498333      0.428571      0.444444   
75%               0.575916               0.748333      0.571429      0.600000   
max               1.000000               1.00000

In [21]:
# Step 10: Remove unnecessary columns and save cleaned dataset
print("\n10. REMOVE UNNECESSARY COLUMNS & SAVE")
print("-" * 70)

# Remove student_id if it exists
cols_to_remove = []
if 'student_id' in df.columns:
    cols_to_remove.append('student_id')
    print(f"Removed column: student_id")

# Remove columns if specified
if cols_to_remove:
    df = df.drop(columns=cols_to_remove)

print(f"\nFinal dataset shape: {df.shape}")
print(f"Total columns: {len(df.columns)}")

# Save cleaned dataset
output_path = Path(r'c:\vscode\CVprojects\kenexaiHackathon\cleaned datasets\cleaned_habits_dataset.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)
print(f"\n✓ Cleaned dataset saved: {output_path}")

# Display final summary
print(f"\n" + "=" * 70)
print("PREPROCESSING COMPLETE - FINAL SUMMARY")
print("=" * 70)
print(f"Dataset ready for ML modeling and merging")
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")

print(f"\n\nAll columns in cleaned dataset:")
for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    print(f"  {i:2d}. {col:40s} ({dtype})")

print(f"\n✓ Pipeline execution completed successfully!")


10. REMOVE UNNECESSARY COLUMNS & SAVE
----------------------------------------------------------------------
Removed column: student_id

Final dataset shape: (80000, 33)
Total columns: 33

✓ Cleaned dataset saved: c:\vscode\CVprojects\kenexaiHackathon\cleaned datasets\cleaned_habits_dataset.csv

PREPROCESSING COMPLETE - FINAL SUMMARY
Dataset ready for ML modeling and merging
Shape: (80000, 33)
Missing values: 0


All columns in cleaned dataset:
   1. age                                      (int64)
   2. gender                                   (category)
   3. major                                    (category)
   4. study_hours_per_day                      (float64)
   5. social_media_hours                       (float64)
   6. netflix_hours                            (float64)
   7. part_time_job                            (int32)
   8. attendance_percentage                    (float64)
   9. sleep_hours                              (float64)
  10. diet_quality                     